## Example 1: pyogrio functionality test 1

In [ ]:
import pyogrio

# Example 1: List supported drivers and formats
print("Testing PyOgrio driver capabilities:")

# List all supported GDAL/OGR drivers.
# list_drivers() maps driver name -> capability string ('r', 'w', 'rw', 'raw', ...).
drivers = pyogrio.list_drivers()
print(f"Supported drivers: {len(drivers)}")

key_formats = ['ESRI Shapefile', 'GeoJSON', 'GPKG', 'FlatGeobuf', 'KML', 'GML']
for fmt in key_formats:
    if fmt in drivers:
        caps = drivers[fmt]
        print(f"  {fmt}: caps={caps!r} (read={'r' in caps}, write={'w' in caps})")

assert 'GeoJSON' in drivers and 'r' in drivers['GeoJSON']

# Get GDAL data path
try:
    gdal_path = pyogrio.get_gdal_data_path()
    print(f"GDAL data path: {gdal_path}")
except Exception:
    print("GDAL data path: Not available")
    raise

# GDAL config options: pyogrio reads ONE option by name (`get_gdal_config_option`) and sets several at once
# (`set_gdal_config_options`); there is no "list all" call. Round-trip a real option, then restore it.
previous_cpl_debug = pyogrio.get_gdal_config_option("CPL_DEBUG")
pyogrio.set_gdal_config_options({"CPL_DEBUG": True})
try:
    assert pyogrio.get_gdal_config_option("CPL_DEBUG") is True
    print(f"GDAL config option CPL_DEBUG: {pyogrio.get_gdal_config_option('CPL_DEBUG')} (was {previous_cpl_debug!r})")
finally:
    pyogrio.set_gdal_config_options({"CPL_DEBUG": previous_cpl_debug})
assert pyogrio.get_gdal_config_option("CPL_DEBUG") == previous_cpl_debug

## Example 2: pyogrio functionality test 2

In [ ]:
import pyogrio
import pandas as pd
import tempfile
import os

# Example 2: Write spatial data to file
print("Testing PyOgrio write operations:")

# Create sample DataFrame with WKT geometry
data = {
    'name': ['Park A', 'Building B', 'Store C'],
    'type': ['park', 'building', 'store']
}
df = pd.DataFrame(data)
df['geometry'] = [
    'POINT (-122.42 37.77)',
    'POINT (-73.94 40.73)',
    'POINT (-87.63 41.87)'
]

print(f"✓ Created DataFrame: {df.shape}")

# Write to GeoJSON
with tempfile.TemporaryDirectory() as tmpdir:
    geojson_path = os.path.join(tmpdir, 'test.geojson')
    pyogrio.write_dataframe(df, geojson_path, driver='GeoJSON')
    print(f"✓ Wrote GeoJSON: {os.path.getsize(geojson_path)} bytes")
    
    # Write to GeoPackage
    gpkg_path = os.path.join(tmpdir, 'test.gpkg')
    pyogrio.write_dataframe(df, gpkg_path, layer='points')
    print(f"✓ Wrote GeoPackage: {os.path.getsize(gpkg_path)} bytes")

## Example 3: pyogrio functionality test 3

In [ ]:
import pyogrio
import pandas as pd
import tempfile
import os

# Example 3: Read spatial data and inspect metadata
print("Testing PyOgrio read operations:")

# Create test data
df = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['A', 'B', 'C'],
    'geometry': [
        'POINT (0 0)',
        'POINT (1 1)', 
        'POINT (2 2)'
    ]
})

with tempfile.TemporaryDirectory() as tmpdir:
    filepath = os.path.join(tmpdir, 'test.geojson')
    pyogrio.write_dataframe(df, filepath)
    
    # Read file info without loading data
    info = pyogrio.read_info(filepath)
    print(f"✓ File info: {info['features']} features")
    print(f"  CRS: {info['crs']}")
    print(f"  Geometry type: {info['geometry_type']}")
    
    # Read bounds
    bounds = pyogrio.read_bounds(filepath)
    print(f"✓ Bounds: {bounds}")
    
    # Read full DataFrame
    loaded = pyogrio.read_dataframe(filepath)
    print(f"✓ Loaded DataFrame: {loaded.shape}")
    print(f"  Columns: {list(loaded.columns)}")

## Example 4: pyogrio functionality test 4

In [ ]:
import pyogrio

# Example 4: Format detection and layer listing
print("Testing PyOgrio format detection:")

# Detect driver from file extension
extensions = ['.shp', '.geojson', '.gpkg', '.fgb', '.kml']
for ext in extensions:
    driver = pyogrio.detect_write_driver(f'test{ext}')
    print(f"✓ {ext} → {driver}")

# List layers in a GeoPackage (multi-layer support)
import pandas as pd
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    gpkg = os.path.join(tmpdir, 'multi.gpkg')
    
    # Write multiple layers
    df1 = pd.DataFrame({'id': [1], 'geometry': ['POINT (0 0)']})
    df2 = pd.DataFrame({'id': [2], 'geometry': ['POINT (1 1)']})
    
    pyogrio.write_dataframe(df1, gpkg, layer='layer1')
    pyogrio.write_dataframe(df2, gpkg, layer='layer2', append=True)
    
    # List layers
    layers = pyogrio.list_layers(gpkg)
    print(f"✓ GeoPackage layers: {layers}")

## Example 5: pyogrio functionality test 5

In [ ]:
import pyogrio
import pandas as pd
import tempfile
import os

# Example 5: Advanced read options
print("Testing PyOgrio advanced read options:")

# Create test data with multiple features
df = pd.DataFrame({
    'name': [f'Feature {i}' for i in range(10)],
    'value': list(range(10)),
    'geometry': [f'POINT ({i} {i})' for i in range(10)]
})

with tempfile.TemporaryDirectory() as tmpdir:
    filepath = os.path.join(tmpdir, 'test.geojson')
    pyogrio.write_dataframe(df, filepath)
    
    # Read with column selection
    subset = pyogrio.read_dataframe(filepath, columns=['name'])
    print(f"✓ Column selection: {list(subset.columns)}")
    
    # Read with feature limit
    limited = pyogrio.read_dataframe(filepath, max_features=3)
    print(f"✓ Limited features: {len(limited)} of 10")
    
    # Read with spatial filter (bbox)
    filtered = pyogrio.read_dataframe(filepath, bbox=(2, 2, 5, 5))
    print(f"✓ Spatial filter: {len(filtered)} features in bbox")
    
    # Skip features
    skipped = pyogrio.read_dataframe(filepath, skip_features=5)
    print(f"✓ Skipped 5: {len(skipped)} remaining")